In [34]:
import pandas as pd
import numpy as np

# 模拟从 ODS 层（原始数据源）导出的未加任何处理的脏日志
data = {
    'user_id': [' u-1029 ', 'U-8812', 'u_9921', 'U-1029', 'sys_test_01'], # 包含空格、大小写不一、分隔符不一、重复用户、测试账号
    'signup_date': ['2023-11-01 14:30:00', '15/12/2023', '2024.01.20', '2023-11-01 14:30:00', '1970-01-01'], # 格式混乱、带有具体时分秒、重复时间、系统幽灵时间
    'monthly_fee': ['$19.99', '19.99', '¥140.00', '19.9', '-9.99'], # 币种混杂、纯数字、退款/黑产数据
    'account_status': ['Active', 'CANCELED', ' pending ', 'Active', 'Admin'], # 大小写不统一、带有隐藏空格、非正常的业务状态
    'user_age': ['28', '35.5', '999', '28', 'NaN'] # 字符串型的数字、带有小数的年龄、老妖精占位符、字符串型的缺失值
}
df_saas_raw = pd.DataFrame(data)
print("📥 原始数据已拉取完毕：")
print(df_saas_raw)

📥 原始数据已拉取完毕：
       user_id          signup_date monthly_fee account_status user_age
0      u-1029   2023-11-01 14:30:00      $19.99         Active       28
1       U-8812           15/12/2023       19.99       CANCELED     35.5
2       u_9921           2024.01.20     ¥140.00       pending       999
3       U-1029  2023-11-01 14:30:00        19.9         Active       28
4  sys_test_01           1970-01-01       -9.99          Admin      NaN


In [ ]:

# 备份数据
df_clean = df_saas_raw.copy()

# 提取user_id
df_clean['user_id'] = df_clean['user_id'].str.strip().str.upper().str.replace('u','U',regex=False).str.replace('_','-',regex=False)
# 删除测试ID行的数据
mask_test_id = df_clean['user_id'].str.contains('test',case=False,na=False)
df_clean = df_clean[~mask_test_id]

# 删除重复ID的行数据
df_clean = df_clean.drop_duplicates(subset=['user_id'],keep='last')


# 处理日期

# 抽样查看
date_type = df_clean['signup_date'].value_counts()


# 第一步：主力格式解析
df_clean['parsed_date'] = pd.to_datetime(df_clean['signup_date'], errors='coerce', format='%Y-%m-%d')

# 第二步：抓取斜杠格式
mask = df_clean['parsed_date'].isna() # 第一次生成掩码，锁定剩下的 4 个
df_clean.loc[mask, 'parsed_date'] = pd.to_datetime(df_clean.loc[mask, 'signup_date'], errors='coerce', format='%Y/%m/%d')

# 第三步：抓取斜杠格式
mask = df_clean['parsed_date'].isna() # 【关键！】刷新掩码，此时第 0 行已经被保护起来了，只锁定剩下的 3 个
df_clean.loc[mask, 'parsed_date'] = pd.to_datetime(df_clean.loc[mask, 'signup_date'], errors='coerce', format='%d/%m/%Y')

# 第四步：抓取.格式
mask = df_clean['parsed_date'].isna() # 【关键！】再次刷新掩码，只锁定剩下的 2 个
df_clean.loc[mask, 'parsed_date'] = pd.to_datetime(df_clean.loc[mask, 'signup_date'], errors='coerce', format='%Y.%m.%d')

# 第五步：抓取带时间的日期格式
mask = df_clean['parsed_date'].isna()
df_clean.loc[mask,'parsed_date'] = pd.to_datetime(df_clean.loc[mask,'signup_date'],errors='coerce',format='%Y-%m-%d %H:%M:%S')

# 剔除异常年份:转换为NaN
mask_abnormal_year = df_clean['parsed_date'].dt.year < 1999
df_clean.loc[mask_abnormal_year,'parsed_date'] = np.nan

# 把所有时间强行抹平到当天的午夜
df_clean['parsed_date'] = df_clean['parsed_date'].dt.normalize()



print("=" * 80)

# 清洗monthly_fee

# 第一步：提取数值和单位并赋予拆分后的字段以列名
extracted = df_clean['monthly_fee'].astype(str).str.extract(r'([^\d\.\-]+)?([\d\.\-]+)')
extracted.columns = ['currency', 'raw_amount']

# 第二步：以美元符号($)填充缺失值
extracted['currency'] = extracted['currency'].fillna('$').str.strip()

# 第三步：将数值部分转换为数字
extracted['raw_amount'] = pd.to_numeric(extracted['raw_amount'], errors='coerce')

# 第四步：建立汇率字典
rate_map ={
    '$':1.0,
    '¥': 0.14
}
# 第五步：使用 map 映射汇率
extracted['exchange_rate'] = extracted['currency'].map(rate_map).fillna(1.0)

# 第六步：相乘得出标准美元金额
df_clean['fee_usd_standard'] = extracted['raw_amount'] * extracted['exchange_rate']


# 第七步：揪出异常数值并填入NaN
mask_abnormal = df_clean['fee_usd_standard'] < 0
df_clean.loc[mask_abnormal,'fee_usd_standard'] = np.nan

# 第八步：压缩内存
df_clean['fee_usd_standard'] = df_clean['fee_usd_standard'].astype('Float32')



# 处理文本列数据account_status

# 第一步：去除空值并转小写
df_clean['account_status'] = df_clean['account_status'].str.strip().str.lower()

# 第二步：业务白名单审判（假设正常状态只有这三种）
valid_statuses = ['active', 'canceled', 'pending']
mask_invalid_status = ~df_clean['account_status'].isin(valid_statuses)

# 第三步： 将不在白名单里的脏状态（如 admin）处决为 NaN
df_clean.loc[mask_invalid_status, 'account_status'] = np.nan


# 清洗年龄列数据user_age —— 优雅链式调用法

df_clean['user_age'] = (
    pd.to_numeric(df_clean['user_age'], errors='coerce')
      .pipe(lambda s: s.where((s >= 0) & (s <= 120)))   # 剔除异常值
      .pipe(lambda s: np.floor(s))
      .astype('Int8')
)


print(df_clean)
# 整理数据表
final_columns = ['user_id', 'parsed_date', 'fee_usd_standard', 'account_status', 'user_age']
df_final = df_clean[final_columns].copy()

# 修改列名
df_final = df_final.rename(columns={
    'parsed_date': 'signup_date',
    'fee_usd_standard': 'monthly_fee'
})
print(df_final)

# 3. 工业级持久化：锁死数据类型
try:
    # 写入 parquet 文件。如果没有安装引擎，可能需要先在终端 pip install pyarrow
    df_final.to_parquet('clean_saas_users.parquet', index=False)
    print("✅ 恭喜，数据清洗完毕，已成功无损落盘至 clean_saas_users.parquet！")
    
    print("\n最后验尸官报告（查看类型是否被完美保留）：")
    print(df_final.dtypes)
except Exception as e:
    print(f"\n❌ 落盘失败！错误信息：{e}")
    print("提示：处理 parquet 需要引擎，请在 Jupyter 新开一个 Cell 运行 !pip install pyarrow 然后重试。")

  user_id          signup_date monthly_fee account_status  user_age  \
1  U-8812           15/12/2023       19.99       canceled        35   
2  U-9921           2024.01.20     ¥140.00        pending      <NA>   
3  U-1029  2023-11-01 14:30:00        19.9         active        28   

  parsed_date  fee_usd_standard  
1  2023-12-15             19.99  
2  2024-01-20              19.6  
3  2023-11-01              19.9  
  user_id signup_date  monthly_fee account_status  user_age
1  U-8812  2023-12-15        19.99       canceled        35
2  U-9921  2024-01-20         19.6        pending      <NA>
3  U-1029  2023-11-01         19.9         active        28
✅ 恭喜，数据清洗完毕，已成功无损落盘至 clean_saas_users.parquet！

最后验尸官报告（查看类型是否被完美保留）：
user_id                   object
signup_date       datetime64[ns]
monthly_fee              Float32
account_status            object
user_age                    Int8
dtype: object


In [45]:
import pandas as pd

# 假设这是明天，下游的数据分析师来读取成果
df_loaded = pd.read_parquet('clean_saas_users.parquet')

print("=== 验尸官最终报告 ===")
print(df_loaded.dtypes)

=== 验尸官最终报告 ===
user_id                   object
signup_date       datetime64[ns]
monthly_fee              Float32
account_status            object
user_age                    Int8
dtype: object
